In [34]:
import os
import re
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import requests
from io import BytesIO
import pandas as pd
import json
from openai import OpenAI
from dotenv import load_dotenv

In [36]:
# Load the BLIP model and processor from Hugging Face, do this once at the start
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

In [37]:
def generate_image_captions(image_folder, sample_size=50):
    """
    Takes a folder of images, generates captions for a random sample, and returns a list of captions.
    """
    captions = []
    image_files = os.listdir(image_folder)[:sample_size]  # Limit to sample size
    
    for file in image_files:
        #print(file)
        raw_image = Image.open(os.path.join(image_folder, file)).convert("RGB")
        
        inputs = processor(raw_image, return_tensors="pt")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        
        captions.append((caption, file))
        
    return captions

In [38]:
test = generate_image_captions("./e-commerce/Apparel/Boys/Images/images_with_product_ids")
print(test)

[("a boy ' s blue striped t - shirt with a sail and a sail", '4203.jpg'), ('t - shirt with print', '38491.jpg'), ('t - shirt with print', '24906.jpg'), ('t - shirt with cat print', '24912.jpg'), ('t - shirt be happy', '36325.jpg'), ('a black t - shirt with a green star on the front', '39002.jpg'), ('t - shirt with a cartoon character', '36319.jpg'), ('a picture of a grey cargo shorts', '39758.jpg'), ('a picture of a denim shorts with a skull on the side', '34056.jpg'), ('a red and white shirt with a small patch on the chest', '34042.jpg'), ('a yellow shorts with a black logo on the side', '31112.jpg'), ('a woman wearing a belted belt and shorts', '40933.jpg'), ("girls ' white t - shirt with yellow and black print", '31106.jpg'), ('jeans with pockets and pockets', '40927.jpg'), ('t - shirt with print', '5486.jpg'), ("a boy ' s shirt with a blue and white checkered pattern", '31099.jpg'), ('a dog wearing a hat and a vest', '3815.jpg'), ('t - shirt with print', '36721.jpg'), ('marvel t - 

In [40]:
# for handling URLs instead of local files:
def generate_image_url_captions(df, url_column, sample_size=50):
    """
    Takes a pandas DataFrame and a column of image URLs, fetches a random sample,
    and returns a list of generated captions.
    """
    captions = []
    valid_df = df.dropna(subset=[url_column])
    sample_df = valid_df.sample(n=min(sample_size, len(valid_df)))
    
    for index, row in sample_df.iterrows():
        url = row[url_column]
        
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            
            raw_image = Image.open(BytesIO(response.content)).convert('RGB')
            
            inputs = processor(raw_image, return_tensors="pt")
            out = model.generate(**inputs)
            caption = processor.decode(out[0], skip_special_tokens=True)
            
            captions.append(caption)
        except Exception as e:
            print(f"Skipped image at index {index} due to error: {e}")
            continue
        
    return captions

In [ ]:
amazon_df = pd.read_csv("./amazon.csv")
captions = generate_image_url_captions(df=amazon_df, url_column="img_link", sample_size=10)

Skipped image at index 1006 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/51ca6eZ+j3L._SY300_SX300_.jpg
Skipped image at index 931 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/51HO3bkK+VS._SY300_SX300_.jpg


In [16]:
print(captions)

[('a black and white wine bottle with a black handle', 'https://m.media-amazon.com/images/I/21OPu5-M3qL._SX300_SY300_QL70_FMwebp_.jpg'), ('a ceiling fan with a black blade', 'https://m.media-amazon.com/images/I/21qojQDoKWL._SX300_SY300_QL70_FMwebp_.jpg'), ('a blender with a blender and blender attachment', 'https://m.media-amazon.com/images/I/41yPeG8kXxL._SX300_SY300_QL70_FMwebp_.jpg'), ('a usb cable with a usb cable attached to it', 'https://m.media-amazon.com/images/I/31l-eZHBfKL._SX300_SY300_QL70_FMwebp_.jpg'), ('a black mixer with a bowl and a mixer attachment', 'https://m.media-amazon.com/images/I/41hYZPZaWfS._SX300_SY300_QL70_FMwebp_.jpg'), ('the apple watch is shown in black', 'https://m.media-amazon.com/images/I/41u0PC4NajL._SX300_SY300_QL70_ML2_.jpg'), ('caso caso caso caso caso caso caso caso caso caso', 'https://m.media-amazon.com/images/I/41LcHKyVl9L._SX300_SY300_QL70_FMwebp_.jpg'), ('anker usb cable usb usb usb usb usb usb usb usb usb usb usb usb usb usb usb usb', 'https:/

### Automatic column type detection

In [21]:
def detect_modalities(df):
    """
    Analyzes a pandas DataFrame to detect which columns likely contain tabular data, text, or image URLs.
    """
    modality_map = {
        'tabular': [],
        'text': [],
        'image_url': []
    }
    image_url_pattern = re.compile(r'^https?://.*\.(?:jpg|jpeg|png|gif|webp).*$', re.IGNORECASE)
    
    for column in df.columns:
        if df[column].dtype == 'object':
            valid_sample = df[column].dropna().astype(str)
            if len(valid_sample) == 0:
                continue
            
            if valid_sample.str.match(image_url_pattern).mean() > 0.5:
                modality_map['image_url'].append(column)
                
            elif valid_sample.str.len().mean() > 50:
                modality_map['text'].append(column)
            else:
                modality_map['tabular'].append(column)
        else:
            modality_map['tabular'].append(column)
            
    return modality_map

In [22]:
modality_info = detect_modalities(amazon_df)
print(modality_info)

{'tabular': ['product_id', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count'], 'text': ['product_name', 'category', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'product_link'], 'image_url': ['img_link']}


### Automatic image file detection

In [23]:
def scan_dataset_directory(base_path):
    """
    Scans a directory to identify csvs and image folders
    """
    dataset_assets = {
        'csv_files': [],
        'image_folders': []
    }
    valid_image_extensions = ('.png', '.jpg', '.jpeg')
    
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith('.csv'):
                dataset_assets['csv_files'].append(os.path.join(root, file))
                
            elif file.lower().endswith(valid_image_extensions):
                if root not in dataset_assets['image_folders']:
                    dataset_assets['image_folders'].append(root)
                    
    return dataset_assets

In [24]:
e_commerce_assets = scan_dataset_directory("./e-commerce")
print(e_commerce_assets)

{'csv_files': ['./e-commerce/fashion.csv'], 'image_folders': ['./e-commerce/Footwear/Men/Images/images_with_product_ids', './e-commerce/Footwear/Women/Images/images_with_product_ids', './e-commerce/Apparel/Girls/Images/images_with_product_ids', './e-commerce/Apparel/Boys/Images/images_with_product_ids']}


### next steps: 
##### test other image caption models
##### build specific profilers for tabular, text, image
##### write llm prompt
##### establish criteria for successful generation

In [29]:
def build_official_autoddg_prompt(dataset_sample, tabular_profile, text_profile, image_profile, data_topic="E-commerce Products", persona="general"):
    # 1. Introduction (From prompts.yaml)
    prompt = f"""Answer the question using the following information.

First, consider the dataset sample:
{dataset_sample}

"""
    # 2. Profile Instruction (From prompts.yaml - Modified for Multimodal)
    prompt += f"""Additionally, the dataset profile is as follows:

[TABULAR STATS]
{tabular_profile}

[UNSTRUCTURED TEXT SAMPLES]
{text_profile}

[VISUAL IMAGE CAPTIONS]
{image_profile}

Based on this multimodal profile, please add sentence(s) to enrich the dataset description.

"""
    # 3. Topic Instruction (From prompts.yaml)
    prompt += f"""Moreover, the dataset topic is: {data_topic}. Based on this topic, please add sentence(s) describing what this dataset can be used for.

"""
    # 4. Closing Instruction (From prompts.yaml - Modified for Personas)
    if persona == "sales":
        prompt += """Question: Based on the information above, provide a dataset description in sentences tailored strictly for a Sales Subject Matter Expert. Focus heavily on the business value, pricing trends, and product visuals. Do not mention technical details like null values, token sizes, or data types. Use only natural, readable sentences.

Answer:"""
    else:
        prompt += """Question: Based on the information above, provide a dataset description in sentences tailored for a Data Scientist. Include a technical breakdown of the specific tabular, visual, and textual modalities. Use only natural, readable sentences.

Answer:"""
        
    return prompt

In [30]:
def profile_tabular_and_semantic(df, tabular_columns):
    """Mimics the summary.py output for the MVP"""
    if not tabular_columns: return "No tabular data detected."
    stats = df[tabular_columns].describe(include='all').to_dict()
    clean_stats = {col: {k: v for k, v in data.items() if pd.notna(v)} for col, data in stats.items()}
    return json.dumps(clean_stats, indent=2)

In [31]:
def profile_text(df, text_columns, sample_size=3):
    """Extracts raw text chunks for the LLM to read"""
    if not text_columns: return "No text data detected."
    text_summary = {}
    for col in text_columns:
        valid_texts = df[col].dropna().astype(str)
        if len(valid_texts) > 0:
            text_summary[col] = valid_texts.sample(min(sample_size, len(valid_texts)), random_state=42).tolist()
    return json.dumps(text_summary, indent=2)

In [43]:
def generate_multimodal_mvp(df, data_topic="E-commerce Products", persona="general"):
    print("1. Dynamically Routing Modalities...")
    modalities = detect_modalities(df) 
    
    print("2. Generating Tabular & Text profiles...")
    try:
        dataset_sample = get_markdown_sample_sorted(df, n=3)
    except NameError:
        dataset_sample = df.head(3).to_markdown() 

    tabular_profile = profile_tabular_and_semantic(df, modalities['tabular'])
    text_profile = profile_text(df, modalities['text'])
    
    print("3. Running Vision Model on detected Image URLs...")
    image_captions_list = []
    # Check if the modality router found any image URL columns
    if modalities['image_url']:
        # Grab the first image column detected
        url_col = modalities['image_url'][0] 
        # Run your actual HuggingFace BLIP pipeline from earlier!
        raw_captions = generate_image_url_captions(df, url_column=url_col, sample_size=5)
        # Extract just the text from the (caption, url) tuples
        image_captions_list = [cap for cap in raw_captions]
        
    image_profile = json.dumps(image_captions_list, indent=2) 
    
    print("4. Building final LLM prompt...")
    final_prompt = build_official_autoddg_prompt(
        dataset_sample=dataset_sample,
        tabular_profile=tabular_profile,
        text_profile=text_profile,
        image_profile=image_profile,
        data_topic=data_topic,
        persona=persona
    )
    
    return final_prompt

# Running the model

In [41]:
load_dotenv('.secrets') 

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [42]:
amazon_df = pd.read_csv("./amazon.csv")

In [48]:
ds_prompt = generate_multimodal_mvp(amazon_df, persona="general")

# 2. Call your LLM API
general_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an assistant for a dataset search engine. Your goal is to improve the readability of dataset descriptions for dataset search engine users."},
        {"role": "user", "content": ds_prompt}
    ],
    temperature=0.0
)

print(general_response.choices[0].message.content)

1. Dynamically Routing Modalities...
2. Generating Tabular & Text profiles...
3. Running Vision Model on detected Image URLs...
Skipped image at index 1371 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/41JWKjRa+PL._SX300_SY300_.jpg
Skipped image at index 19 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/51v-2Nzr+ML._SY300_SX300_.jpg
4. Building final LLM prompt...
This dataset comprises a comprehensive collection of e-commerce product information, specifically focusing on electronic accessories such as charging cables and adapters. It contains 1,465 entries, each characterized by various attributes including product ID, name, category, pricing details, ratings, and user reviews. 

The tabular data is structured with key columns such as `product_id`, `product_name`, `category`, `discounted_price`, `actual_price`, `discount_percentage`, `rating`, and `ratin

In [46]:
sales_prompt = generate_multimodal_mvp(amazon_df, persona="sales")

# 2. Call the OpenAI API
sales_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an assistant for a dataset search engine. Your goal is to improve the readability of dataset descriptions for dataset search engine users."},
        {"role": "user", "content": sales_prompt}
    ],
    temperature=0.0
)

sales_description = sales_response.choices[0].message.content
print(sales_description)

1. Dynamically Routing Modalities...
2. Generating Tabular & Text profiles...
3. Running Vision Model on detected Image URLs...
Skipped image at index 1328 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T2/images/I/419H62Is66L._SX300_SY300_QL70_FMwebp_.jpg
Skipped image at index 170 due to error: 400 Client Error: Bad Request for url: https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/31IdziegWVL._SX300_SY300_QL70_FMwebp_.jpg
4. Building final LLM prompt...
This dataset offers a comprehensive collection of e-commerce products, primarily focusing on accessories for computers and mobile devices. It features detailed descriptions of various charging cables, highlighting their compatibility, durability, and fast-charging capabilities. Each product entry includes essential pricing information, showcasing discounted prices alongside actual prices, which can help identify pricing trends and promotional strategies.

The dataset als

In [47]:
def generate_search_focused_description(initial_description, topic="E-commerce Electronic Accessories"):
    """
    Takes a human-readable dataset description and expands it into a 
    keyword-heavy outline for search engine indexing.
    """
    
    # The exact template from the original AutoDDG prompts.yaml
    search_template = """
    Dataset Overview:
    - Please keep the exact initial description of the dataset as shown in beginning the prompt.

    Key Themes or Topics:
    - Central focus on a broad area of interest (e.g., urban planning, socio-economic factors, environmental analysis).
    - Data spans multiple subtopics or related areas that contribute to a holistic understanding of the primary theme.
    Example:
    - theme1/topic1
    - theme2/topic2
    - theme3/topic3

    Applications and Use Cases:
    - Facilitates analysis for professionals, policymakers, researchers, or stakeholders.
    - Useful for specific applications, such as planning, engineering, policy formulation, or statistical modeling.
    - Enables insights into patterns, trends, and relationships relevant to the domain.
    Example:
    - application1/usecase1
    - application2/usecase2
    - application3/usecase3

    Concepts and Synonyms:
    - Includes related concepts, terms, and variations to ensure comprehensive coverage of the topic.
    - Synonyms and alternative phrases improve searchability and retrieval effectiveness.
    Example:
    - concept1/synonym1
    - concept2/synonym2
    - concept3/synonym3

    Keywords and Themes:
    - Lists relevant keywords and themes for indexing, categorization, and enhancing discoverability.
    - Keywords reflect the dataset's content, scope, and relevance to the domain.
    Example:
    - keyword1
    - keyword2
    - keyword3

    Additional Context:
    - Highlights the dataset's relevance to specific challenges or questions in the domain.
    - May emphasize its value for interdisciplinary applications or integration with related datasets.
    Example:
    - context1
    - context2
    - context3
    """

    # The exact user prompt from the original AutoDDG prompts.yaml
    user_prompt = f"""You are given a dataset about the topic {topic}, with the following initial description:

{initial_description}

Please expand the description by including the exact topic. Additionally, add as many related concepts, synonyms, and relevant terms as possible based on the initial description and the topic.

Unlike the initial description, which is focused on presentation and readability, the expanded description is intended to be indexed at the backend of a dataset search engine to improve searchability.

Therefore, focus less on readability and more on including all relevant terms related to the topic. Make sure to include any variations of the key terms and concepts that could help improve retrieval in search results.

Please follow the structure of the following example template:
{search_template}
"""

    # Call the API using their exact Search-Focused system message
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an assistant for a dataset search engine. Your goal is to improve the performance of the dataset search engine for keyword queries."},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0
    )
    
    return response.choices[0].message.content